<a href="https://colab.research.google.com/github/Iditc/log-anomaly-detection/blob/main/notebooks/02_full_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Define paths
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/log-anomaly-detection')
DATA_PROC = BASE / 'data' / 'processed'
RESULTS = BASE / 'results'

RESULTS.mkdir(parents=True, exist_ok=True)
print("Ready.")

Ready.


In [9]:
# Load full parsed dataset
df = pd.read_parquet(DATA_PROC / "hdfs_full_parsed.parquet")
print(f"Shape: {df.shape}")

Shape: (11175629, 4)


Suspicious templates:

template 32 — 96% anomaly! Almost every line with this template is an anomaly
templates 6, 11, 12 — ~63% anomaly

In [10]:
# Show all templates with their frequency and label distribution
template_stats = df.groupby("template_id").agg(
    count=("Label", "size"),
    anomaly_count=("Label", lambda x: (x == "Anomaly").sum()),
    anomaly_rate=("Label", lambda x: (x == "Anomaly").mean())
).sort_values("count", ascending=False)

print(template_stats.head(20).to_string())


               count  anomaly_count  anomaly_rate
template_id                                      
1            1723232          45945      0.026662
5            1719741          40256      0.023408
4            1706514          31845      0.018661
17           1402047          35473      0.025301
31           1396174          31923      0.022865
49            859939          13132      0.015271
36            680085          14350      0.021100
34            356207           8259      0.023186
38            351415           8950      0.025468
48            288998           6729      0.023284
35            229113           7541      0.032914
3             166704           4577      0.027456
13            120036           3129      0.026067
2              56950           2568      0.045092
47             42062           1671      0.039727
10             35249           1221      0.034639
6               7097           4480      0.631253
11              6837           4371      0.639315


In [11]:
# Show all 55 templates — sample one log line per template
for tid in [32, 6, 11, 12]:
    sample = df[df["template_id"] == tid]["log_line"].iloc[0]
    rate = template_stats.loc[tid, "anomaly_rate"]
    print(f"Template {tid} (anomaly rate: {rate:.1%}):")
    print(f"  {sample}\n")

Template 32 (anomaly rate: 96.0%):
  081109 213815 19 WARN dfs.FSDataset: Unexpected error trying to delete block blk_-1067131609371010449. BlockInfo not found in volumeMap.

Template 6 (anomaly rate: 63.1%):
  081109 203521 143 INFO dfs.DataNode$DataXceiver: Received block blk_-1608999687919862906 src: /10.251.215.16:52002 dest: /10.251.215.16:50010 of size 91178

Template 11 (anomaly rate: 63.9%):
  081109 203530 19 INFO dfs.FSNamesystem: BLOCK* ask 10.251.31.5:50010 to replicate blk_-1608999687919862906 to datanode(s) 10.251.90.64:50010

Template 12 (anomaly rate: 63.9%):
  081109 203531 19 INFO dfs.DataNode: 10.251.31.5:50010 Starting thread to transfer block blk_-1608999687919862906 to 10.251.90.64:50010



This code groups all log lines by block_id, counts how many times each template appears in each block, and adds the label (Normal/Anomaly). The result is one row per block with 55 template-count features.

In [12]:
# Count how many times each template appears in each block
count_vectors = df.groupby(["block_id", "template_id"]).size().unstack(fill_value=0)

# Add prefix to column names
count_vectors.columns = [f"t_{c}" for c in count_vectors.columns]

# Add labels
block_labels = df.groupby("block_id")["Label"].first()
count_vectors["Label"] = block_labels

print(f"Shape: {count_vectors.shape}")
print(f"\nLabel distribution:")
print(count_vectors["Label"].value_counts())
print(f"\nFirst 5 rows:")
print(count_vectors.head())

Shape: (575061, 56)

Label distribution:
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64

First 5 rows:
                          t_1  t_2  t_3  t_4  t_5  t_6  t_7  t_8  t_9  t_10  \
block_id                                                                      
blk_-1000002529962039464    3    0    0    3    3    0    0    0    0     0   
blk_-100000266894974466     3    0    0    3    3    0    0    0    0     0   
blk_-1000007292892887521    3    0    0    3    3    0    0    0    0     0   
blk_-1000014584150379967    3    0    0    3    3    0    0    0    0     0   
blk_-1000028658773048709    3    0    0    3    3    0    0    0    0     0   

                          ...  t_47  t_48  t_49  t_50  t_51  t_52  t_53  t_54  \
block_id                  ...                                                   
blk_-1000002529962039464  ...     0     1     3     0     0     0     0     0   
blk_-100000266894974466   ...     0     0     0     0     0     0     0     0  

Save this table to parquet so we can use it in the baseline notebook:

In [13]:
# Save block-level count vectors to parquet for model training
count_vectors.to_parquet(DATA_PROC / "hdfs_block_vectors.parquet")
print("Saved to hdfs_block_vectors.parquet")

Saved to hdfs_block_vectors.parquet
